In [1]:
import importlib
import utils
importlib.reload(utils) 
from utils import *

In [2]:
kaggle_cred_file = r'E:\env\KAGGLE_KEY\kaggle.json'
load_kaggle_creds(kaggle_cred_file)

In [4]:
competition = 'playground-series-s4e12'
download_competition_data(competition)

Unzipping data files


In [11]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [39]:
train_file = 'data/train.csv'
train_df = pd.read_csv(train_file)
train_df['Policy Start Date'] = pd.to_datetime(train_df['Policy Start Date'])

In [40]:
train_df.head()

,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type,Premium Amount
0,0,19.0,Female,10049.0,Married,1.0,Bachelor's,Self-Employed,22.598761,Urban,Premium,2.0,17.0,372.0,5.0,2023-12-23 15:21:39.134960,Poor,No,Weekly,House,2869.0
1,1,39.0,Female,31678.0,Divorced,3.0,Master's,NaN,15.569731,Rural,Comprehensive,1.0,12.0,694.0,2.0,2023-06-12 15:21:39.111551,Average,Yes,Monthly,House,1483.0
2,2,23.0,Male,25602.0,Divorced,3.0,High School,Self-Employed,47.177549,Suburban,Premium,1.0,14.0,NaN,3.0,2023-09-30 15:21:39.221386,Good,Yes,Weekly,House,567.0
3,3,21.0,Male,141855.0,Married,2.0,Bachelor's,NaN,10.938144,Rural,Basic,1.0,0.0,367.0,1.0,2024-06-12 15:21:39.226954,Poor,Yes,Daily,Apartment,765.0
4,4,21.0,Male,39651.0,Single,1.0,Bachelor's,Self-Employed,20.376094,Rural,Premium,0.0,8.0,598.0,4.0,2021-12-01 15:21:39.252145,Poor,Yes,Weekly,House,2022.0


In [41]:
train_df.columns

Index(['id', 'Age', 'Gender', 'Annual Income', 'Marital Status',
       'Number of Dependents', 'Education Level', 'Occupation', 'Health Score',
       'Location', 'Policy Type', 'Previous Claims', 'Vehicle Age',
       'Credit Score', 'Insurance Duration', 'Policy Start Date',
       'Customer Feedback', 'Smoking Status', 'Exercise Frequency',
       'Property Type', 'Premium Amount'],
      dtype='object')

In [47]:
class DataPipeline:
    def __init__(self):
        self.cat_cols = ['Gender', 'Marital Status', 'Education Level', 'Occupation', 'Location', 'Policy Type',
                         'Customer Feedback', 'Smoking Status', 'Exercise Frequency', 'Property Type']
        self.int_cols =  ['Age','Number of Dependents', 'Previous Claims', 'Vehicle Age', 'Credit Score','Insurance Duration']
        self.float_cols = ['Health Score', 'Annual Income']
        #self.df = df.copy()
        self.mode_dict = {}
        self.mean_dict = {}
        self.scaler = None
        
    def imputer(self, df):
        for col in self.cat_cols + self.int_cols:
            self.mode_dict[col] = df[col].mode()[0]
        for col in self.float_cols:
            self.mean_dict[col] = float(df[col].mean())
        return {**self.mode_dict, **self.mean_dict}            
 
    def allowed_values(self, df):
        values_seen = {}
        for col in self.cat_cols + self.int_cols:
            values_seen[col] = df[col].dropna().unique().tolist()
        return values_seen
    def fit(self, df):
        self.mode_dict = self.imputer(df)
        self.allowed_values = self.allowed_values(df)
        #self.scaler
        return self
    
    def transform(self, df=None):
        df = df.copy()
        for col in self.cat_cols + self.int_cols:
            df[col] = df[col].fillna(self.mode_dict[col])
            df.loc[~df[col].isin(self.allowed_values[col]), col] = self.mode_dict[col]

        for col in self.cat_cols:
            df[col] = df[col].astype('category') 
        for col in self.float_cols:
            df[col] = df[col].fillna(self.mean_dict[col])
        return df

In [48]:
pipe = DataPipeline().fit(train_df)

In [49]:
train_df_transformed = pipe.transform(train_df)

In [50]:
train_df_transformed.head()

,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type,Premium Amount
0,0,19.0,Female,10049.0,Married,1.0,Bachelor's,Self-Employed,22.598761,Urban,Premium,2.0,17.0,372.0,5.0,2023-12-23 15:21:39.134960,Poor,No,Weekly,House,2869.0
1,1,39.0,Female,31678.0,Divorced,3.0,Master's,Employed,15.569731,Rural,Comprehensive,1.0,12.0,694.0,2.0,2023-06-12 15:21:39.111551,Average,Yes,Monthly,House,1483.0
2,2,23.0,Male,25602.0,Divorced,3.0,High School,Self-Employed,47.177549,Suburban,Premium,1.0,14.0,434.0,3.0,2023-09-30 15:21:39.221386,Good,Yes,Weekly,House,567.0
3,3,21.0,Male,141855.0,Married,2.0,Bachelor's,Employed,10.938144,Rural,Basic,1.0,0.0,367.0,1.0,2024-06-12 15:21:39.226954,Poor,Yes,Daily,Apartment,765.0
4,4,21.0,Male,39651.0,Single,1.0,Bachelor's,Self-Employed,20.376094,Rural,Premium,0.0,8.0,598.0,4.0,2021-12-01 15:21:39.252145,Poor,Yes,Weekly,House,2022.0


In [52]:
feature_set = pipe.cat_cols + pipe.int_cols + pipe.float_cols

In [53]:
feature_set

['Gender',
 'Marital Status',
 'Education Level',
 'Occupation',
 'Location',
 'Policy Type',
 'Customer Feedback',
 'Smoking Status',
 'Exercise Frequency',
 'Property Type',
 'Age',
 'Number of Dependents',
 'Previous Claims',
 'Vehicle Age',
 'Credit Score',
 'Insurance Duration',
 'Health Score',
 'Annual Income']

In [60]:
X = train_df_transformed[feature_set]
y = train_df_transformed['Premium Amount']

In [61]:
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [62]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [63]:
model = xgb.XGBRegressor(
    objective="reg:squarederror",  # Standard objective for regression
    n_estimators=100,              # Number of trees
    learning_rate=0.1,             # Step size shrinkage
    max_depth=3,                   # Maximum depth of a tree
    random_state=42,
    enable_categorical = True
)


In [65]:
%%time

model.fit(X_train, y_train)

CPU times: total: 4.34 s
Wall time: 1.1 s


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [66]:
from sklearn.metrics import mean_squared_error,mean_absolute_error

In [67]:
# Make predictions
y_pred = model.predict(X_test)

# Evaluate performance using Mean Squared Error (MSE)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse:.4f}")

mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae:.4f}")

Mean Squared Error: 730734.0222
Mean Absolute Error: 658.7716


In [68]:
from sklearn.model_selection import KFold
import numpy as np

In [71]:
# Define cross-validation strategy
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Store RMSE scores for each fold
rmse_train, rmse_test = [], []
mae_train, mae_test = [], []

xgb_reg = model
# Perform manual cross-validation
for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx] 
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx] 

    # Train the model
    xgb_reg.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    rmse_train.append(train_rmse)
    rmse_test.append(test_rmse)
    mae_train.append(train_mae)
    mae_test.append(test_mae)

    print(f"Fold {fold}: Train RMSE = {train_rmse:.4f}, Test RMSE = {test_rmse:.4f}")
    print(f"         Train MAE  = {train_mae:.4f}, Test MAE  = {test_mae:.4f}\n")

# Average scores across folds
print("Average Train RMSE:", np.mean(rmse_train))
print("Average Test RMSE:", np.mean(rmse_test))
print("Average Train MAE:", np.mean(mae_train))
print("Average Test MAE:", np.mean(mae_test))

# Train the final model
xgb_reg.fit(X,y)

Fold 1: Train RMSE = 854.4648, Test RMSE = 854.2383
         Train MAE  = 657.9050, Test MAE  = 657.9593

Fold 2: Train RMSE = 854.9553, Test RMSE = 853.9840
         Train MAE  = 658.3506, Test MAE  = 658.1357

Fold 3: Train RMSE = 854.0070, Test RMSE = 856.3382
         Train MAE  = 657.4652, Test MAE  = 658.2763

Fold 4: Train RMSE = 854.6833, Test RMSE = 854.6011
         Train MAE  = 658.2288, Test MAE  = 658.5948

Fold 5: Train RMSE = 854.8553, Test RMSE = 855.0810
         Train MAE  = 658.5571, Test MAE  = 658.4654

Average Train RMSE: 854.59314138086
Average Test RMSE: 854.8485184371068
Average Train MAE: 658.1013448970032
Average Test MAE: 658.286306976064


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [73]:
test_file = 'data/test.csv'
test_df = pd.read_csv(test_file)


In [74]:
test_df.head()

,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type
0,1200000,28.0,Female,2310.0,NaN,4.0,Bachelor's,Self-Employed,7.657981,Rural,Basic,NaN,19.0,NaN,1.0,2023-06-04 15:21:39.245086,Poor,Yes,Weekly,House
1,1200001,31.0,Female,126031.0,Married,2.0,Master's,Self-Employed,13.381379,Suburban,Premium,NaN,14.0,372.0,8.0,2024-04-22 15:21:39.224915,Good,Yes,Rarely,Apartment
2,1200002,47.0,Female,17092.0,Divorced,0.0,PhD,Unemployed,24.354527,Urban,Comprehensive,NaN,16.0,819.0,9.0,2023-04-05 15:21:39.134960,Average,Yes,Monthly,Condo
3,1200003,28.0,Female,30424.0,Divorced,3.0,PhD,Self-Employed,5.136225,Suburban,Comprehensive,1.0,3.0,770.0,5.0,2023-10-25 15:21:39.134960,Poor,Yes,Daily,House
4,1200004,24.0,Male,10863.0,Divorced,2.0,High School,Unemployed,11.844155,Suburban,Premium,NaN,14.0,755.0,7.0,2021-11-26 15:21:39.259788,Average,No,Weekly,House


In [75]:
ids = test_df['id'].values
test_df = test_df.drop(columns=['id'])

In [77]:
test_df_transformed = pipe.transform(test_df)
feature_set = pipe.cat_cols + pipe.int_cols + pipe.float_cols
X_test = test_df_transformed[feature_set]

In [78]:
y_pred = xgb_reg.predict(X_test)

In [79]:
submission_df = pd.DataFrame({'id':ids, 'Premium Amount': y_pred})

In [80]:
sumission_file = "my_submission_20250225121500.csv"
submission_df.to_csv(sumission_file, index=False) 

In [81]:
upload_submission(competition, sumission_file, 'my first submission')

In [87]:
mean_absolute_error.__name__

'mean_absolute_error'